# Trustpilot Ingestion Notebook (Data Solutions API)

In [0]:
import hashlib
import time
from datetime import datetime, timezone
from typing import Dict, List, Optional

import requests
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
CONFIG = {
    "api_base_url": "https://datasolutions.trustpilot.com/v1",
    "legacy_api_base_url": "https://api.trustpilot.com/v1",
    "secret_scope": "trustpilot",
    "secret_key": "api_key",
    "per_page": 100,
    "max_retries": 5,
    "backoff_seconds": 2,
    "catalog": "avant_users",
    "schema": "kaley_ubellacker",
    "bronze_table": "trustpilot_reviews_bronze",
    "csv_output": "/Volumes/avant_users/kaley_ubellacker/sentiment_analysis",
}

COMPANIES = [
    {"company": "avant", "domain": "www.avant.com"},
    {"company": "mission_lane", "domain": "www.missionlane.com"},
    {"company": "merrick_bank", "domain": "www.merrickbank.com"},
    {"company": "onemain_financial", "domain": "www.onemainfinancial.com"},
    {"company": "concora", "domain": "concoracredit.com"},
    {"company": "indigo", "domain": "www.indigocard.com"},
    {"company": "credit_one", "domain": "www.creditonebank.com"},
]

In [0]:
def get_api_key() -> str:
    return dbutils.secrets.get(scope=CONFIG["secret_scope"], key=CONFIG["secret_key"])


def request_with_retry(url: str, params: Dict, headers: Dict) -> Dict:
    last_response = None
    for attempt in range(1, CONFIG["max_retries"] + 1):
        r = requests.get(url, params=params, headers=headers, timeout=30)
        last_response = r
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(CONFIG["backoff_seconds"] * (2 ** (attempt - 1)))
            continue
        break

    if last_response is not None and last_response.status_code == 403:
        body = (last_response.text or "")[:400]
        raise PermissionError(
            "403 from Trustpilot API. Verify Data Solutions API key + product access. "
            "Display API cannot access full service reviews; Insights API is required. "
            f"URL={last_response.url}; body_sample={body}"
        )
    if last_response is not None:
        last_response.raise_for_status()
    raise RuntimeError(f"Failed API call after retries: {url}")


def get_business_unit_id(domain: str, api_key: str) -> str:
    # Data Solutions documented flow: GET /v1/business-units?domain={domain}
    candidates = [
        (f"{CONFIG['api_base_url']}/business-units", {"domain": domain}),
        (f"{CONFIG['legacy_api_base_url']}/business-units/find", {"name": domain}),
    ]
    for url, params in candidates:
        try:
            payload = request_with_retry(url=url, params=params, headers={"apikey": api_key})
            if isinstance(payload, dict) and payload.get("id"):
                return payload["id"]
            if isinstance(payload, dict) and payload.get("businessUnits"):
                return payload["businessUnits"][0]["id"]
            if isinstance(payload, list) and payload:
                return payload[0]["id"]
        except Exception:
            continue

    raise ValueError(f"Could not resolve business unit id for domain={domain}")


def parse_reviews(company: str, domain: str, business_unit_id: str, payload: Dict) -> List[Dict]:
    out = []
    now_utc = datetime.now(timezone.utc).isoformat()
    rows = payload.get("reviews", payload.get("data", []))
    for row in rows:
        consumer = row.get("consumer", {}) or {}
        display_name = consumer.get("displayName", "")
        reviewer_hash = hashlib.sha256(display_name.encode("utf-8")).hexdigest() if display_name else None
        out.append({
            "review_id": row.get("id") or row.get("reviewId"),
            "company": company,
            "domain": domain,
            "business_unit_id": business_unit_id,
            "rating": row.get("stars") or row.get("rating"),
            "title": row.get("title"),
            "text": row.get("text") or row.get("comment"),
            "language": row.get("language"),
            "created_at": row.get("createdAt") or row.get("createdDate"),
            "updated_at": row.get("updatedAt") or row.get("updatedDate"),
            "consumer_country": consumer.get("countryCode"),
            "consumer_hash": reviewer_hash,
            "source": "trustpilot",
            "ingested_at": now_utc,
        })
    return out


def fetch_company_reviews(company: str, domain: str, api_key: str) -> List[Dict]:
    business_unit_id = get_business_unit_id(domain, api_key)
    print(f"Resolved business unit id for {company}: {business_unit_id}")

    all_reviews: List[Dict] = []
    page = 1
    page_token: Optional[str] = None

    while True:
        params = {"perPage": CONFIG["per_page"], "page": page}
        if page_token:
            params["pageToken"] = page_token

        payload = request_with_retry(
            url=f"{CONFIG['api_base_url']}/business-units/{business_unit_id}/reviews",
            params=params,
            headers={"apikey": api_key},
        )

        rows = parse_reviews(company, domain, business_unit_id, payload)
        if not rows:
            break
        all_reviews.extend(rows)

        next_token = payload.get("nextPageToken") or payload.get("pageToken")
        has_next = payload.get("links", {}).get("next") if isinstance(payload.get("links"), dict) else None
        if next_token:
            page_token = next_token
            time.sleep(0.25)
            continue
        if has_next:
            page += 1
            time.sleep(0.25)
            continue
        if len(rows) < CONFIG["per_page"]:
            break
        page += 1

    return all_reviews

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CONFIG['catalog']}.{CONFIG['schema']}")
api_key = get_api_key()
all_rows: List[Dict] = []

for item in COMPANIES:
    print(f"Fetching: {item['company']} ({item['domain']})")
    all_rows.extend(fetch_company_reviews(item["company"], item["domain"], api_key))

schema = T.StructType([
    T.StructField("review_id", T.StringType()),
    T.StructField("company", T.StringType()),
    T.StructField("domain", T.StringType()),
    T.StructField("business_unit_id", T.StringType()),
    T.StructField("rating", T.IntegerType()),
    T.StructField("title", T.StringType()),
    T.StructField("text", T.StringType()),
    T.StructField("language", T.StringType()),
    T.StructField("created_at", T.StringType()),
    T.StructField("updated_at", T.StringType()),
    T.StructField("consumer_country", T.StringType()),
    T.StructField("consumer_hash", T.StringType()),
    T.StructField("source", T.StringType()),
    T.StructField("ingested_at", T.StringType()),
])

raw_df = spark.createDataFrame(all_rows, schema=schema)
clean_df = (
    raw_df.withColumn("created_ts", F.to_timestamp("created_at"))
    .withColumn("updated_ts", F.to_timestamp("updated_at"))
    .dropDuplicates(["review_id", "company"])
)

clean_df.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable(
    f"{CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['bronze_table']}"
)
clean_df.coalesce(1).write.mode("overwrite").option("header", True).csv(CONFIG["csv_output"])

spark.sql(f"OPTIMIZE {CONFIG['catalog']}.{CONFIG['schema']}.{CONFIG['bronze_table']} ZORDER BY (company, created_ts)")
print(f"Ingested {clean_df.count()} reviews")